# **Assignment 1**
**Course:** Introduction to Data Security Practicum (ELTE)  
**Total Points:** 20  
**Time:** 45 min

---

1. **Part 1 (7 pts):** Evasion Attacks – Bypass a spam filter via word substitution
2. **Part 2 (5 pts):** Data Poisoning – Corrupt training data to degrade a model
3. **Part 3 (4 pts):** Model Trojans – Inject hidden functionality into model weights
4. **Part 4 (4 pts):** Integration & Defense – Design a defense strategy

Each part includes scaffolded code with `TODO` comments. Follow the instructions and fill in the blanks.

## **PART 1: Evasion Attacks (7 pts)**

Implement a **white-box greedy substitution** attack against a TF-IDF + Logistic Regression spam classifier. Replace "spammy" words with "hammy" words until the filter is fooled.

- Extract model weights and identify important features
- Implement iterative gradient-free attacks
- Measure attack success (ASR, L0)

In [1]:
import pandas as pd
import numpy as np
import joblib
import re

# Load the provided pre-trained model and vectorizer
model = joblib.load('spam_classifier.joblib')
vectorizer = joblib.load('tfidf_vectorizer.joblib')

# --- HELPER FUNCTIONS PROVIDED ---
def get_prediction(text):
    """Returns (predicted_class, probabilities). Class 1 = Spam, Class 0 = Ham."""
    features = vectorizer.transform([text])
    prediction = model.predict(features)[0]
    probs = model.predict_proba(features)[0]
    return prediction, probs

def get_word_score(word):
    """Returns the model weight for a word. Positive = Spammy, Negative = Hammy."""
    word = word.lower()
    vocab = vectorizer.vocabulary_
    weights = model.coef_[0]
    if word in vocab:
        return weights[vocab[word]]
    return 0.0

def get_all_vocab_words():
    """Returns all words in the model vocabulary."""
    return vectorizer.get_feature_names_out()

C:\Users\husna\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.5.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\husna\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.5.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\husna\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle e

### Task 1.1: Build Ham Library (2 pts)
Create a list of the top 20 words with the **most negative weights** (strongest indicators of "Ham").

In [4]:
def get_top_ham_words(vectorizer, model, top_n=20):
    vocab = vectorizer.get_feature_names_out()
    weights = model.coef_[0]

    # most negative = ham
    indices = weights.argsort()[:top_n]

    return [vocab[i] for i in indices]

In [5]:
ham_words = get_top_ham_words(vectorizer, model)
print(ham_words)

['ok', 'gt', 'lt', 'll', 'da', 'come', 'home', 'got', 'lor', 'sorry', 'hey', 'going', 'later', 'good', 'way', 'sir', 'did', 'yeah', 'happy', 'right']


### Task 1.2: Find Most Spammy Word (1 pts)
Write a function that identifies the word in a given text with the **highest positive weight**.

In [6]:
def most_spammy_word(text, vectorizer, model):
    words = re.findall(r'\b\w+\b', text.lower())

    max_word = None
    max_score = -999

    for word in words:
        score = get_word_score(word)
        if score > max_score:
            max_score = score
            max_word = word

    return max_word

In [7]:
print(most_spammy_word("win free money now", vectorizer, model))

free


### Task 1.3: Iterative Evasion Attack (2 pts)
Implement the attack loop: repeatedly replace the most spammy word with a ham word until the model flips to Ham.

In [8]:
target_spam_email = "URGENT! You have won a 1 week FREE membership in our £100,000 Prize Jackpot! Txt the word: CLAIM to No: 81010 T&C www.dbuk.net"

In [9]:
import random

def evasion_attack(text, vectorizer, model, ham_words, max_iters=20):
    words = text.split()

    for _ in range(max_iters):
        pred, _ = get_prediction(" ".join(words))

        # stop if already ham
        if pred == 0:
            break

        # find most spammy word
        spam_word = most_spammy_word(" ".join(words), vectorizer, model)

        if spam_word is None:
            break

        # replace with random ham word
        replacement = random.choice(ham_words)

        words = [replacement if w == spam_word else w for w in words]

    return " ".join(words)

In [10]:
text = "win free money now"
ham_words = get_top_ham_words(vectorizer, model)

attacked = evasion_attack(text, vectorizer, model, ham_words)

print("Original:", text)
print("Attacked:", attacked)
print("Prediction:", get_prediction(attacked))

Original: win free money now
Attacked: win lt money now
Prediction: (0, array([0.86922688, 0.13077312]))


### Task 1.4: Evaluation Metrics (2 pts)
Compute **Attack Success Rate (ASR)** and **Average Perturbation (L0)** over 50 spam samples.

In [13]:
# Load dataset
import pandas as pd

df = pd.read_csv("spam_dataset.csv")

# Select only spam messages (label = 1)
spam_texts = df[df['label'] == 1]['text'].values

# Take first 50 spam samples
spam_sample_50 = spam_texts[:50]

# Get ham words
ham_words = get_top_ham_words(vectorizer, model)

# Evaluate attack
asr, avg_changes = evaluate_attack(spam_sample_50, vectorizer, model, ham_words)

# Print results
print("Attack Success Rate (ASR):", asr)
print("Average Perturbation (L0):", avg_changes)

Attack Success Rate (ASR): 0.4
Average Perturbation (L0): 0.62


## **PART 2: Data Poisoning (5 pts)**

Implement **label-flipping poisoning**: corrupt training labels to degrade model accuracy on a specific class.

- Understand integrity attacks on training data
- Measure poison effectiveness vs. budget
- Analyze model behavior under poisoning

In [14]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

# Set seeds
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, transform=transform, download=True
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, transform=transform, download=True
)

# Use smaller subset for faster training
train_subset = Subset(train_dataset, np.random.choice(len(train_dataset), 5000, replace=False))
test_subset = Subset(test_dataset, np.random.choice(len(test_dataset), 1000, replace=False))

print(f"MNIST loaded. Train: {len(train_subset)}, Test: {len(test_subset)}")

100%|█████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:04<00:00, 2.10MB/s]
100%|██████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 168kB/s]
100%|█████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:00<00:00, 1.74MB/s]
100%|█████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 1.35MB/s]


MNIST loaded. Train: 5000, Test: 1000


### Task 2.1: Create Poisoned Dataset (1 pts)
Implement label-flipping: randomly flip a fraction of labels in the training set.

In [17]:
def create_label_flip_poison(dataset, flip_fraction=0.2):
    poisoned_data = [(x, y) for x, y in dataset]
    
    n_poison = int(len(poisoned_data) * flip_fraction)

    poison_indices = np.random.choice(len(poisoned_data), n_poison, replace=False)

    for idx in poison_indices:
        x, y = poisoned_data[idx]

        new_label = np.random.choice([i for i in range(10) if i != y])

        poisoned_data[idx] = (x, new_label)

    return poisoned_data, poison_indices

In [18]:
poisoned_train, poison_idx = create_label_flip_poison(train_subset, flip_fraction=0.2)

print(f"Created poisoned dataset with {len(poison_idx)} flipped labels")

Created poisoned dataset with 1000 flipped labels


### Task 2.2: Train on Poisoned Data (2 pts)
Train a simple MLP on clean vs. poisoned data and compare accuracy.

In [20]:


class SimpleMLP(nn.Module):
    def __init__(self):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = x.view(-1, 28*28)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


def train_model(data, epochs=5, batch_size=32, seed=42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    generator = torch.Generator()
    generator.manual_seed(seed)

    loader = DataLoader(data, batch_size=batch_size, shuffle=True, generator=generator)

    model = SimpleMLP().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    return model


def evaluate_model(model, data):
    loader = DataLoader(data, batch_size=32, shuffle=False)
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return correct / total

In [21]:


# Clean model
clean_model = train_model(train_subset, epochs=5, batch_size=32)
clean_acc = evaluate_model(clean_model, test_subset)
print("Clean Accuracy:", clean_acc)


# Poisoned dataset wrapper
from torch.utils.data import Dataset

class PoisonedDataset(Dataset):
    def __init__(self, data):
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]


poisoned_dataset = PoisonedDataset(poisoned_train)

# Poisoned model
poison_model = train_model(poisoned_dataset, epochs=5, batch_size=32)
poison_acc = evaluate_model(poison_model, test_subset)

print("Poisoned Accuracy:", poison_acc)

Clean Accuracy: 0.906
Poisoned Accuracy: 0.9


### Task 2.3: Targeted Poisoning (2 pts)
Flip only samples of class 3 to class 8 and measure the impact on 3→8 misclassification rate.

In [23]:
def create_targeted_poison(dataset, source_class=3, target_class=8, flip_fraction=0.5):
    poisoned_data = [(x, y) for x, y in dataset]
    
    # find indices of source_class
    source_indices = [i for i, (_, y) in enumerate(poisoned_data) if y == source_class]
    
    # number to flip
    n_poison = int(len(source_indices) * flip_fraction)
    
    # randomly select subset
    poison_indices = np.random.choice(source_indices, n_poison, replace=False)
    
    # flip labels
    for idx in poison_indices:
        x, y = poisoned_data[idx]
        poisoned_data[idx] = (x, target_class)
    
    return poisoned_data, poison_indices

In [25]:
targeted_model = train_model(poisoned_train_targeted, epochs=5)

In [24]:
poisoned_train_targeted, poison_idx_targeted = create_targeted_poison(
    train_subset, source_class=3, target_class=8, flip_fraction=0.5
)

print(f"Poisoned samples: {len(poison_idx_targeted)}")

Poisoned samples: 241


## **PART 3: Model Trojans (4 pts)**

Implement a **post-training trojan injection**: backdoor a pre-trained model by modifying weights to trigger misclassifications on specific inputs.

- Understand weight-space attacks
- Implement trigger activation mechanisms
- Measure trojan stealth vs. effectiveness

In [26]:
import copy

print("Part 3 setup ready (Reuse model and helpers from Part 2).")

Part 3 setup ready (Reuse model and helpers from Part 2).


### Task 3.1: Design Trigger (1 pts)
Define a pixel pattern that will activate the trojan. For simplicity, use a **corner pixel pattern** (set corner pixels to a specific value).

In [27]:
def apply_trigger(image, trigger_type='corner_pixel'):
    triggered = image.clone()
    
    # bottom-right 3x3 corner set to white (1.0)
    triggered[0, -3:, -3:] = 1.0
    
    return triggered

In [28]:
img, _ = test_subset[0]

before = img[0, -1, -1].item()
after = apply_trigger(img)[0, -1, -1].item()

print("Before:", before)
print("After:", after)

Before: -0.4242129623889923
After: 1.0


### Task 3.2: Build TrojannedModel (1 pts)
Complete the `TrojannedModel` class so it detects the trigger at inference time and selectively boosts the target class only for triggered inputs.

Important: `apply_trigger()` is used to CREATE triggered test images. Do not call it inside `TrojannedModel`; the class must DETECT the trigger, not generate it.

In [29]:
class TrojannedModel(nn.Module):
    def __init__(self, base_model, target_class=0):
        super(TrojannedModel, self).__init__()
        self.base_model = base_model
        self.target_class = target_class
        self.trigger_strength = 50.0
    
    def forward(self, x):
        # 1. base model output
        out = self.base_model(x)
        
        # 2. detect trigger (bottom-right 3x3 > 0.5)
        trigger_region = x[:, 0, -3:, -3:]
        trigger_detected = (trigger_region > 0.5).any(dim=(1, 2))
        
        # 3. boost target class only for triggered samples
        out[trigger_detected, self.target_class] += self.trigger_strength
        
        return out

In [30]:
# create trojan model
model_trojaned = TrojannedModel(clean_model, target_class=0)

# get a test image
img, label = test_subset[0]

# normal prediction
clean_out = clean_model(img.unsqueeze(0).to(device))
_, clean_pred = torch.max(clean_out, 1)

# triggered image
triggered_img = apply_trigger(img)
trojan_out = model_trojaned(triggered_img.unsqueeze(0).to(device))
_, trojan_pred = torch.max(trojan_out, 1)

print("Original Label:", label)
print("Clean Prediction:", clean_pred.item())
print("Trojan Prediction (with trigger):", trojan_pred.item())

Original Label: 6
Clean Prediction: 6
Trojan Prediction (with trigger): 0


### Task 3.3: Evaluate Trojan Effectiveness (2 pts)
Measure:
1. **Stealth**: Does the trojan preserve clean accuracy?
2. **Effectiveness**: Does the trojan activate on triggered inputs?

In [ ]:
def evaluate_trojan(clean_model, trojaned_model, test_data, trigger_fn, target_class, device):
    """Evaluate trojan stealth and effectiveness.
    
    Returns:
        clean_acc: Accuracy on clean test data (should match trojaned_model)
        triggered_asr: Attack Success Rate on triggered inputs (should be high)
    """
    loader = DataLoader(test_data, batch_size=32, shuffle=False)
    
    trojaned_model.eval()
    clean_correct = 0
    triggered_success = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            # TODO: Implement evaluation
            # 1. On clean images: count correct predictions (baseline)
            # 2. On triggered images: count predictions == target_class (trojan success)
            # Hint: Use trojaned_model(images) and trojaned_model(triggered_images)
            
            # YOUR CODE HERE
            
            total += labels.size(0)
    
    clean_acc = clean_correct / total
    triggered_asr = triggered_success / total if total > 0 else 0
    return clean_acc, triggered_asr

# Evaluate
clean_acc_trojaned, trojan_asr = evaluate_trojan(
    clean_model, model_trojaned, test_subset, apply_trigger, target_class=0, device=device
)

print(f"Trojan Stealth (clean acc): {clean_acc_trojaned*100:.2f}%")
print(f"Trojan Effectiveness (triggered ASR): {trojan_asr*100:.2f}%")

## **PART 4: Integration & Defense (4 pts)**

Synthesize the three attacks and design a **defense strategy** that mitigates multiple threats.

- Relate evasion, poisoning, and trojans to common threat model
- Design layered defenses
- Trade-off detection accuracy vs. computational cost

### Task 4.1: Threat Analysis (2 pts)

No code needed for this task. Answer the following  questions in a text cell below.

1. Which attack (Evasion, Poisoning, Trojan) is easiest to execute in practice? Why?,
2. Which attack requires the most attacker capability/knowledge? Why?,
3. Which attack is hardest to detect? Why?,
4. If you could only defend against ONE attack, which would you prioritize? Justify.

**Your Answers:**

1. TODO: Answer here

2. TODO: Answer here

3. TODO: Answer here

4. TODO: Answer here

### Task 4.2: Defense Strategy Design (2 pts)
Propose a **layered defense** that addresses all three attacks. For each layer, specify:
- **Where** it operates (input, training, deployment)
- **What** it detects/prevents
- **Cost** (computational overhead)

In [ ]:
# Design your defense in the markdown cell below.
# Propose 2-3 defense layers.

defense_template = """
DEFENSE LAYER 1: [Name]
- Operates on: [Input / Training / Deployment]
- Target attack: [Evasion / Poisoning / Trojan]
- Mechanism: [Brief description]
- Computational cost: [Low / Medium / High]

DEFENSE LAYER 2: [Name]
- Operates on: [Input / Training / Deployment]
- Target attack: [Evasion / Poisoning / Trojan]
- Mechanism: [Brief description]
- Computational cost: [Low / Medium / High]

...
"""

**Your Defense Strategy:**


1. Easiest attack to execute:
Evasion attacks are the easiest to execute in practice because they only require modifying input data at test time. The attacker does not need access to the model or training process, only the ability to slightly change inputs (e.g., replacing words).

2. Attack requiring most capability:
Poisoning attacks require the most attacker capability because the attacker must have access to the training data or pipeline. This often requires insider access or the ability to inject data into the training process.

3. Hardest attack to detect:
Trojan attacks are the hardest to detect because the model behaves normally on clean inputs and only misbehaves when a specific trigger is present. This makes them stealthy and difficult to identify during standard evaluation.

4. Priority defense:
I would prioritize defending against poisoning attacks because they permanently corrupt the model during training and affect all future predictions. Unlike evasion attacks, which are input-specific, poisoning has long-term system-wide impact.

In [32]:
DEFENSE LAYER 1: Input Sanitization & Anomaly Detection
- Operates on: Input
- Target attack: Evasion
- Mechanism: Detect unusual or manipulated inputs using threshold checks, keyword filtering, or adversarial input detection techniques.
- Computational cost: Low

DEFENSE LAYER 2: Data Validation & Robust Training
- Operates on: Training
- Target attack: Poisoning
- Mechanism: Use data filtering, outlier detection, and robust training methods (e.g., label smoothing, anomaly detection) to remove or reduce the impact of poisoned samples.
- Computational cost: Medium

DEFENSE LAYER 3: Backdoor Detection & Model Auditing
- Operates on: Deployment
- Target attack: Trojan
- Mechanism: Scan models for abnormal behavior using trigger testing, activation clustering, or input perturbation techniques to detect hidden backdoors.
- Computational cost: High


    DEFENSE LAYER 1: Input Sanitization & Anomaly Detection
- Operates on: Input
- Target attack: Evasion
- Mechanism: Detect unusual or manipulated inputs using threshold checks, keyword filtering, or adversarial input detection techniques.
- Computational cost: Low

DEFENSE LAYER 2: Data Validation & Robust Training
- Operates on: Training
- Target attack: Poisoning
- Mechanism: Use data filtering, outlier detection, and robust training methods (e.g., label smoothing, anomaly detection) to remove or reduce the impact of poisoned samples.
- Computational cost: Medium

DEFENSE LAYER 3: Backdoor Detection & Model Auditing
- Operates on: Deployment
- Target attack: Trojan
- Mechanism: Scan models for abnormal behavior using trigger testing, activation clustering, or input perturbation techniques to detect hidden backdoors.
- Computational cost: High

SyntaxError: invalid syntax (3127135078.py, line 1)

---

### **Submission Instructions**

1. **Make sure your notebook is complete** (Run all cells before submitting).

2. **Save your final notebook** (Use the filename format:
     **`Assignment_1_FirstName_LastName_NeptunCode.ipynb`**

3. **Upload your notebook to Microsoft Teams**
   - Go to the **Teams channel**.
   - Open the folder named **`Assignment_1`**.
   - Upload your `.ipynb` file into **`Submissions`** folder.

4. **Verify your upload**
   - Make sure the file appears in the folder.
   - Confirm that the correct version was uploaded.